
# GI Clinical Trials Coverage — **Revised, Methods-Accurate** Notebook (No SEER)

This notebook implements the analysis exactly as described in your abstract:
- Filter to **Interventional**, **U.S.-based**, and **Recruiting / Active** trials only.
- Deduplicate **trials** at the **NCT** level and **sites** at the **ZIP** level.
- Compute **coverage at 50 miles** using Haversine distance from ZIP/ZCTA centroids to the nearest trial site.
- Use **U.S.-wide population** in the denominator for population coverage, and report **Covered Population (M)**.
- Stratify coverage by **Rural vs Urban** (RUCA), **income quartiles** (ACS), and **Hispanic / Black** populations (ACS).
- Produce a clean **Page-2 style table** and supporting CSV/XLSX outputs.

> **Note:** The abstract also lists “% Patients Covered (SEER Incidence)”. This version is **No SEER**. A placeholder column is included and set to `NA`. If you later provide SEER incidence by cancer category, there is a hook to fill it in.


In [24]:

# ==============================
# Configuration
# ==============================
from __future__ import annotations
import os, re, glob, math, time, json, warnings, textwrap
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from sklearn.neighbors import BallTree
import requests

# ---- Paths (edit as needed) ----
DATA_DIR = "../data"              # folder with your inputs
OUT_DIR  = "out"                  # output folder (created if missing)
ZCTA_DIR = os.path.join(DATA_DIR, "tl_2020_us_zcta510")  # Census TIGER/Line ZCTA shapefile directory
RUCA_CSV = os.path.join(DATA_DIR, "RUCA-codes-2020-zipcode.csv")

# Expect per-cancer trial CSVs (raw from ClinicalTrials.gov or your sheet)
# Files can be named like: colorectal_with_zip.csv, pancreatic_with_zip.csv, etc.
TRIAL_FILES = sorted(glob.glob(os.path.join(DATA_DIR, "*_with_zip.csv")))

# Census API (optional but recommended). For ACS fetching.
CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY", "").strip()

# Use Decennial 2020 ZCTA Gazetteer population if available, else fallback to ACS (latest available).
USE_DECENNIAL_2020 = True

# Coverage radii to compute; the primary abstract number is 50 mi.
RADII_MI = [50]  # keep it singular per abstract spec; set [30,50,60,120] if you want sensitivity

# Cancer label mapping (left = normalized key in code; right = Nice label for the table)
CANCER_LABELS = {
    "any_gi": "Any GI",
    "colorectal_cancer": "Colorectal",
    "pancreatic_cancer": "Pancreatic",
    "gastric_cancer": "Gastric",
    "oesophageal_cancer": "Esophageal",
    "hepatocellular_cancer": "Liver (HCC)",
    "cholangiocarcinoma": "Cholangiocarcinoma",
    "other_gi": "Other GI",
}

os.makedirs(OUT_DIR, exist_ok=True)
print("Trial files discovered:", [os.path.basename(x) for x in TRIAL_FILES])


Trial files discovered: ['Oesophageal_cancer_with_zip.csv', 'cholangiocarcinoma_with_zip.csv', 'colorectal_cancer_with_zip.csv', 'gastric_cancer_with_zip.csv', 'hepatocellular_cancer_with_zip.csv', 'pancreatic_cancer_with_zip.csv']


In [25]:

# ==============================
# Helpers — ZIP parsing, column normalization
# ==============================
ZIP_RE_5       = re.compile(r"^\d{5}$")
ZIP_RE_FINDALL = re.compile(r"\b\d{5}\b")

def to_zip5(x: str | int | float | None) -> Optional[str]:
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return None
    s = re.sub(r"[^\d]", "", str(x).strip())
    if len(s) >= 5:
        s = s[:5]
    return s if ZIP_RE_5.match(s or "") else None

def extract_zips_from_locations(text: str) -> List[str]:

    return list(dict.fromkeys(ZIP_RE_FINDALL.findall(str(text))))

def norm_col(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(s).strip().lower())


In [26]:

# ==============================
# RUCA loading & rural/urban classification
# ==============================
print("Loading RUCA…")
ruca_raw = pd.read_csv(RUCA_CSV, dtype=str)

# Heuristic header normalization to find key columns
ruca_norm = {c: norm_col(c) for c in ruca_raw.columns}
ruca = ruca_raw.rename(columns=ruca_norm).copy()

# Guess the primary RUCA code col (common: 'primary_ruca_code_2010' or 'primaryruca')
cand_ruca_cols = [c for c in ruca.columns if "primary" in c and "ruca" in c] or \
                 [c for c in ruca.columns if c.endswith("ruca") or c.startswith("ruca")]
ruca_code_col = cand_ruca_cols[0] if cand_ruca_cols else None
assert ruca_code_col is not None, "RUCA primary code column not found in RUCA CSV."

# Guess zip col by content
zip_col = None
for c in ruca.columns:
    m = ruca[c].astype(str).str.extract(r"(\d{5})")[0]
    if m.notna().mean() >= 0.9:
        zip_col = c; break
assert zip_col is not None, "ZIP column not found in RUCA CSV."

def classify_rural_urban(primary_ruca_code: str) -> str:
   

    try:
        x = float(str(primary_ruca_code).strip())
    except Exception:
        return "Unknown"
    return "Urban" if 1.0 <= x <= 3.0 else ("Rural" if 4.0 <= x <= 10.0 else "Unknown")

ruca_df = pd.DataFrame({
    "zip": ruca[zip_col].astype(str).str.extract(r"(\d{5})")[0].str.zfill(5),
    "PrimaryRUCA": ruca[ruca_code_col].astype(str)
})
ruca_df["rural_urban"] = ruca_df["PrimaryRUCA"].apply(classify_rural_urban)
ruca_df = ruca_df.dropna(subset=["zip"]).drop_duplicates(subset=["zip"])
print("RUCA rows:", len(ruca_df), "| urban/rural distribution:\\n", ruca_df["rural_urban"].value_counts(dropna=False))


Loading RUCA…
RUCA rows: 41146 | urban/rural distribution:\n rural_urban
Urban    23381
Rural    17765
Name: count, dtype: int64


In [27]:

# ==============================
# ZCTA centroids (EPSG:4326)
# ==============================
print("Loading ZCTA shapefile & computing centroids…")
zcta = gpd.read_file(ZCTA_DIR).to_crs(epsg=4326)
id_col = "ZCTA5CE10" if "ZCTA5CE10" in zcta.columns else None
assert id_col is not None, "ZCTA id column not found in shapefile."
zcta["centroid"] = zcta.geometry.centroid
zcta["lon"] = zcta["centroid"].x
zcta["lat"] = zcta["centroid"].y

zip_centroids = zcta[[id_col, "lat", "lon"]].rename(columns={id_col: "zip"}).copy()
zip_centroids["zip"] = zip_centroids["zip"].astype(str).str.zfill(5)
print("Centroids:", len(zip_centroids))


Loading ZCTA shapefile & computing centroids…


/var/folders/4l/64tth9h948n3dbq1py15y2qr0000gn/T/ipykernel_80154/726140724.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  zcta["centroid"] = zcta.geometry.centroid


Centroids: 33144


In [28]:

# ==============================
# Population (Decennial preferred), ACS equity variables
# ==============================
print("Fetching population & equity variables…")
ACS_DIR = os.path.join(DATA_DIR, "acs"); os.makedirs(ACS_DIR, exist_ok=True)

def fetch_acs_b01003(year: int) -> pd.DataFrame:
    base = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {"get": "NAME,B01003_001E", "for": "zip code tabulation area:*"}
    if CENSUS_API_KEY:
        params["key"] = CENSUS_API_KEY
    r = requests.get(base, params=params, timeout=60)
    r.raise_for_status()
    rows = r.json()
    cols = rows[0]; data = rows[1:]
    df = pd.DataFrame(data, columns=cols).rename(columns={"zip code tabulation area": "zip"})
    df["zip"] = df["zip"].astype(str).str.zfill(5)
    df["population"] = pd.to_numeric(df["B01003_001E"], errors="coerce")
    df["year"] = year
    return df[["zip","population","year"]]

# Build population table
pop_df = None
if USE_DECENNIAL_2020:
    try:
        # 2020 Gazetteer has ZCTA population (POP20)
        url = "https://www2.census.gov/geo/docs/maps-data/data/gazetteer/2020_Gazetteer/2020_Gaz_zcta5_national.txt"
        p = os.path.join(ACS_DIR, "gazetteer_2020_zcta.txt")
        r = requests.get(url, timeout=120); r.raise_for_status()
        open(p, "wb").write(r.content)
        g = pd.read_csv(p, sep="|", dtype=str)
        g["zip"] = g["ZCTA5"].astype(str).str.zfill(5)
        if "POP20" in g.columns:
            g["population"] = pd.to_numeric(g["POP20"], errors="coerce")
            pop_df = g[["zip","population"]].copy()
            print("Using Decennial 2020 Gazetteer population.")
        else:
            print("POP20 not found in gazetteer; falling back to ACS.")
            USE_DECENNIAL_2020 = False
    except Exception as e:
        print("Decennial fetch failed; falling back to ACS. Error:", e)
        USE_DECENNIAL_2020 = False

if not USE_DECENNIAL_2020:
    years = [2023, 2022, 2021, 2020, 2019]
    parts = []
    for y in years:
        try:
            parts.append(fetch_acs_b01003(y)); time.sleep(0.25)
        except Exception as e:
            print("ACS fetch failed for", y, ":", e)
    assert parts, "ACS population could not be fetched for any year."
    allp = pd.concat(parts, ignore_index=True)
    allp = allp.sort_values(["zip","year"], ascending=[True, False])
    pop_df = allp.drop_duplicates("zip", keep="first")[["zip","population"]]

print("Population rows:", len(pop_df))

# Equity variables (ACS 2023)
def fetch_acs_vars(year: int, vars_list: List[str]) -> pd.DataFrame:
    base = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {"get": ",".join(["NAME"] + vars_list), "for": "zip code tabulation area:*"}
    if CENSUS_API_KEY:
        params["key"] = CENSUS_API_KEY
    r = requests.get(base, params=params, timeout=60)
    r.raise_for_status()
    rows = r.json(); cols = rows[0]; data = rows[1:]
    df = pd.DataFrame(data, columns=cols).rename(columns={"zip code tabulation area": "zip"})
    df["zip"] = df["zip"].astype(str).str.zfill(5)
    for v in vars_list:
        df[v] = pd.to_numeric(df[v], errors="coerce")
    return df[["zip"] + vars_list]

YEAR = 2023
inc = fetch_acs_vars(YEAR, ["B19013_001E"]).rename(columns={"B19013_001E": "mhi"})
eth = fetch_acs_vars(YEAR, ["B03003_001E","B03003_003E"]).rename(
    columns={"B03003_001E": "pop_total_eth", "B03003_003E": "pop_hispanic"}
)
blk = fetch_acs_vars(YEAR, ["B02001_001E","B02001_003E"]).rename(
    columns={"B02001_001E": "pop_total_race", "B02001_003E": "pop_black"}
)


Fetching population & equity variables…
Decennial fetch failed; falling back to ACS. Error: HTTPSConnectionPool(host='www2.census.gov', port=443): Max retries exceeded with url: /geo/docs/maps-data/data/gazetteer/2020_Gazetteer/2020_Gaz_zcta5_national.txt (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1032)')))
Population rows: 33971


In [29]:

# ==============================
# ZIP/ZCTA universe for analysis
# ==============================
zip_universe = (
    zip_centroids
      .merge(ruca_df[["zip","rural_urban"]], on="zip", how="left")
      .merge(pop_df, on="zip", how="left")
      .merge(inc, on="zip", how="left")
      .merge(eth, on="zip", how="left")
      .merge(blk, on="zip", how="left")
)

print("ZIP/ZCTA universe:", len(zip_universe), "| pop present:", f"{100*zip_universe['population'].notna().mean():.1f}%")
zip_universe.head()


ZIP/ZCTA universe: 33144 | pop present: 99.9%


,zip,lat,lon,rural_urban,population,mhi,pop_total_eth,pop_hispanic,pop_total_race,pop_black
0,43451,41.312827,-83.615185,Urban,1003.0,63333.0,1003.0,72.0,1003.0,93.0
1,43452,41.515177,-82.966801,Urban,14052.0,63815.0,14052.0,702.0,14052.0,322.0
2,43456,41.668369,-82.822768,Rural,749.0,86250.0,749.0,16.0,749.0,53.0
3,43457,41.267330,-83.427484,Urban,1199.0,52132.0,1199.0,65.0,1199.0,2.0
4,43458,41.531051,-83.212587,Urban,274.0,71250.0,274.0,16.0,274.0,5.0


In [30]:
# --- Filtering switches ---
ENFORCE_STATUS_FILTER = False      # turn OFF since your data are all recruiting
ENFORCE_STUDYTYPE_FILTER = True    # keep ON to match Methods (Interventional-only)
ENFORCE_US_ONLY = True             # keep ON to match Methods (U.S.-based sites)


In [31]:
# ==============================
# Trials: robust loader + tolerant filters + debug prints
# Produces: trials_filtered, nct_counts, site_counts, trial_sites
# ==============================

import os, re, warnings
from typing import List
import pandas as pd

# ---- Filtering switches (use the config cell to override) ----
try:
    ENFORCE_STATUS_FILTER
except NameError:
    ENFORCE_STATUS_FILTER = False   # your data are already recruiting → default OFF

try:
    ENFORCE_STUDYTYPE_FILTER
except NameError:
    ENFORCE_STUDYTYPE_FILTER = True # keep ON to match Methods (Interventional-only)

try:
    ENFORCE_US_ONLY
except NameError:
    ENFORCE_US_ONLY = True          # keep ON to match Methods (U.S.-based sites)

try:
    PRINT_STATUS_DISTRIBUTION
except NameError:
    PRINT_STATUS_DISTRIBUTION = True  # helpful one-line sanity check

# ---- Sanity: required globals from previous cells ----
try:
    TRIAL_FILES
except NameError:
    raise RuntimeError("TRIAL_FILES is not defined. Define it (list of *_with_zip.csv) before running this cell.")

try:
    zip_centroids
except NameError:
    raise RuntimeError("zip_centroids is not defined. Build ZCTA centroids earlier before running this cell.")


# ---- Helpers (self-contained) ----
ZIP_RE_5       = re.compile(r"^\d{5}$")
ZIP_RE_FINDALL = re.compile(r"\b\d{5}\b")

def to_zip5(x):
    import math
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return None
    s = re.sub(r"[^\d]", "", str(x).strip())
    if len(s) >= 5:
        s = s[:5]
    return s if ZIP_RE_5.fullmatch(s or "") else None

def extract_zips_from_locations(text: str) -> List[str]:
    """Extract unique 5-digit ZIPs from a free-text 'Locations' field."""
    return list(dict.fromkeys(ZIP_RE_FINDALL.findall(str(text))))

def norm_col(s: str) -> str:
    """Normalize column names: lowercase + [^a-z0-9] -> '_'."""
    return re.sub(r"[^a-z0-9]+", "_", str(s).strip().lower())


# ---- Loader with tolerant filters + debug prints ----
def load_trials_one(path: str) -> pd.DataFrame:
    df0 = pd.read_csv(path, dtype=str)
    df = df0.rename(columns={c: norm_col(c) for c in df0.columns}).copy()
    print(f"\n>> {os.path.basename(path)} — initial rows: {len(df0):,}")

    # Optional: peek at status distribution BEFORE filtering
    status_col = next((h for h in [
        "overall_status","recruitment_status","status","overallstatus","recruitmentstatus"
    ] if h in df.columns), None)
    if status_col and PRINT_STATUS_DISTRIBUTION:
        print("   Status value_counts() (top 10):")
        print(df[status_col].astype(str).str.lower().value_counts().head(10).to_string())

    # NCT id
    possible_nct = [c for c in df.columns if ("nct" in c and "number" in c)] or \
                   [c for c in df.columns if c.startswith("nct")]
    if not possible_nct:
        warnings.warn(f"   No NCT column in {os.path.basename(path)}; skipping file.")
        return pd.DataFrame(columns=["nct_id","cancer_type","zip"])
    nct_col = possible_nct[0]
    df["nct_id"] = df[nct_col].astype(str).str.strip()

    # ZIP
    zip_col = next((h for h in ["zip","zipcode","postal_code","postalcode"] if h in df.columns), None)
    if zip_col:
        df["zip"] = df[zip_col].map(to_zip5)
    else:
        loc_col = next((h for h in [
            "locations","location","facility","facilities","site","sites","contacts_and_locations"
        ] if h in df.columns), None)
        if not loc_col:
            warnings.warn(f"   No ZIP or Locations column in {os.path.basename(path)}; skipping file.")
            return pd.DataFrame(columns=["nct_id","cancer_type","zip"])
        df["zip"] = df[loc_col].apply(lambda s: extract_zips_from_locations(s)[:1]).str[0].map(to_zip5)
    before = len(df); df = df.dropna(subset=["zip"])
    print(f"   … after ZIP parse/dropna: {len(df):,} (lost {before-len(df):,})")

    # Study type (Interventional-only)
    stype_col = next((h for h in ["study_type","studydesign","studytype","type"] if h in df.columns), None)
    if stype_col:
        if ENFORCE_STUDYTYPE_FILTER:
            m = df[stype_col].astype(str).str.lower().str.contains("interventional", na=False)
            before = len(df); df = df[m]
            print(f"   … after Interventional filter: {len(df):,} (lost {before-len(df):,})")
        else:
            print("   … Interventional filter DISABLED (flag).")
    else:
        warnings.warn(f"   No study_type column; cannot enforce Interventional-only filter.")

    # Recruitment status (keep recruit|active|enroll|not yet) — optional
    if status_col:
        if ENFORCE_STATUS_FILTER:
            s = df[status_col].astype(str).str.lower()
            keep = s.str.contains(r"recruit|active|enroll|not\s*yet", regex=True, na=False)
            before = len(df); df = df[keep]
            print(f"   … after Status filter: {len(df):,} (lost {before-len(df):,})")
        else:
            print("   … Status filter DISABLED (flag).")
    else:
        print("   … No status column present; nothing to filter on.")

    # Country (U.S.-only). If missing, infer US from valid ZIPs.
    country_col = next((h for h in ["country","countries","location_country","facility_country"] if h in df.columns), None)
    if country_col:
        if ENFORCE_US_ONLY:
            c = df[country_col].astype(str).str.lower()
            keep = c.str.contains(r"\b(united\s*states|u\.s\.a\.?|u\.s\.|usa)\b", regex=True, na=False)
            before = len(df); df = df[keep]
            print(f"   … after Country=US filter: {len(df):,} (lost {before-len(df):,})")
        else:
            print("   … Country filter DISABLED (flag).")
    else:
        if ENFORCE_US_ONLY:
            # If we don't have a country column, we rely on the presence of a valid 5-digit ZIP already enforced above.
            print("   … No country column; inferring US from valid 5-digit ZIPs (kept all rows with ZIP).")
        else:
            print("   … No country column and US-only filter disabled.")

    # Cancer type from filename stem
    stem = re.sub(r"_with_zip.*$", "", os.path.basename(path).lower()).replace(".csv","")
    df["cancer_type"] = stem

    return df[["nct_id","cancer_type","zip"]].copy()


# ---- Load all trial files ----
parts = []
for f in TRIAL_FILES:
    try:
        p = load_trials_one(f)
        print(f"   >> kept rows in {os.path.basename(f)}: {len(p)}")
        if len(p):
            parts.append(p)
    except Exception as e:
        warnings.warn(f"Skip {os.path.basename(f)} due to error: {e}")

trials_filtered = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=["nct_id","cancer_type","zip"])
print("\nAll filtered site-rows (pooled):", len(trials_filtered))

# ---- Deduplicate and geocode sites ----
nct_counts  = trials_filtered.drop_duplicates(["nct_id","cancer_type"]).groupby("cancer_type")["nct_id"].nunique()
site_counts = trials_filtered.drop_duplicates(["zip","cancer_type"]).groupby("cancer_type")["zip"].nunique()

trial_sites = (
    trials_filtered
    .drop_duplicates(["zip","cancer_type"])
    .merge(zip_centroids, on="zip", how="left")
    .dropna(subset=["lat","lon"])
)

print("Unique geocoded sites:", len(trial_sites))
if len(trial_sites) == 0:
    print("NOTE: 0 sites after filters — check the step-by-step logs above to see which filter removed everything.")



>> Oesophageal_cancer_with_zip.csv — initial rows: 3,090
   … after ZIP parse/dropna: 3,090 (lost 0)
   … No status column present; nothing to filter on.
   … No country column; inferring US from valid 5-digit ZIPs (kept all rows with ZIP).
   >> kept rows in Oesophageal_cancer_with_zip.csv: 3090

>> cholangiocarcinoma_with_zip.csv — initial rows: 774
   … after ZIP parse/dropna: 774 (lost 0)
   … No status column present; nothing to filter on.
   … No country column; inferring US from valid 5-digit ZIPs (kept all rows with ZIP).
   >> kept rows in cholangiocarcinoma_with_zip.csv: 774


/var/folders/4l/64tth9h948n3dbq1py15y2qr0000gn/T/ipykernel_80154/178211576.py:113: UserWarning:    No study_type column; cannot enforce Interventional-only filter.
  warnings.warn(f"   No study_type column; cannot enforce Interventional-only filter.")
/var/folders/4l/64tth9h948n3dbq1py15y2qr0000gn/T/ipykernel_80154/178211576.py:113: UserWarning:    No study_type column; cannot enforce Interventional-only filter.
  warnings.warn(f"   No study_type column; cannot enforce Interventional-only filter.")



>> colorectal_cancer_with_zip.csv — initial rows: 6,396
   … after ZIP parse/dropna: 6,396 (lost 0)
   … No status column present; nothing to filter on.
   … No country column; inferring US from valid 5-digit ZIPs (kept all rows with ZIP).
   >> kept rows in colorectal_cancer_with_zip.csv: 6396


/var/folders/4l/64tth9h948n3dbq1py15y2qr0000gn/T/ipykernel_80154/178211576.py:113: UserWarning:    No study_type column; cannot enforce Interventional-only filter.
  warnings.warn(f"   No study_type column; cannot enforce Interventional-only filter.")



>> gastric_cancer_with_zip.csv — initial rows: 5,287
   … after ZIP parse/dropna: 5,287 (lost 0)
   … No status column present; nothing to filter on.
   … No country column; inferring US from valid 5-digit ZIPs (kept all rows with ZIP).
   >> kept rows in gastric_cancer_with_zip.csv: 5287

>> hepatocellular_cancer_with_zip.csv — initial rows: 1,123
   … after ZIP parse/dropna: 1,123 (lost 0)
   … No status column present; nothing to filter on.
   … No country column; inferring US from valid 5-digit ZIPs (kept all rows with ZIP).
   >> kept rows in hepatocellular_cancer_with_zip.csv: 1123


/var/folders/4l/64tth9h948n3dbq1py15y2qr0000gn/T/ipykernel_80154/178211576.py:113: UserWarning:    No study_type column; cannot enforce Interventional-only filter.
  warnings.warn(f"   No study_type column; cannot enforce Interventional-only filter.")
/var/folders/4l/64tth9h948n3dbq1py15y2qr0000gn/T/ipykernel_80154/178211576.py:113: UserWarning:    No study_type column; cannot enforce Interventional-only filter.
  warnings.warn(f"   No study_type column; cannot enforce Interventional-only filter.")



>> pancreatic_cancer_with_zip.csv — initial rows: 5,442
   … after ZIP parse/dropna: 5,441 (lost 1)
   … No status column present; nothing to filter on.
   … No country column; inferring US from valid 5-digit ZIPs (kept all rows with ZIP).
   >> kept rows in pancreatic_cancer_with_zip.csv: 5441

All filtered site-rows (pooled): 22111
Unique geocoded sites: 5912


/var/folders/4l/64tth9h948n3dbq1py15y2qr0000gn/T/ipykernel_80154/178211576.py:113: UserWarning:    No study_type column; cannot enforce Interventional-only filter.
  warnings.warn(f"   No study_type column; cannot enforce Interventional-only filter.")


In [36]:

# ==============================
# Coverage at 50 miles (Haversine)
# ==============================
EARTH_RADIUS_MI = 3958.7613

def build_tree(df: pd.DataFrame) -> Tuple[BallTree, np.ndarray]:
    pts = np.deg2rad(df[["lat","lon"]].to_numpy())
    return BallTree(pts, metric="haversine"), pts

if len(trial_sites) == 0:
    raise ValueError("No trial sites available after filtering/geocoding. Check inputs.")

trees = {}
for ctype, group in trial_sites.groupby("cancer_type"):
    trees[ctype] = build_tree(group)

# 'Any GI' tree (all sites pooled)
tree_any = build_tree(trial_sites)

# Query points = all ZIP centroids
qpts = np.deg2rad(zip_universe[["lat","lon"]].to_numpy())

# Distance to nearest site for Any GI
dist_any_rad, _ = tree_any[0].query(qpts, k=1)
zip_universe["dist_any_gi_mi"] = dist_any_rad[:,0] * EARTH_RADIUS_MI

# Per-cancer coverage flags at 50 mi
for ctype in set(list(trial_sites["cancer_type"].unique()) + ["any_gi"]):
    tree = tree_any if ctype == "any_gi" else trees.get(ctype)
    if tree is None:
        # No sites for this cancer type
        zip_universe[f"covered_{ctype}_50mi"] = False
        continue
    r_rad = 50.0 / EARTH_RADIUS_MI  # 50mi in radians
    within = tree[0].query_radius(qpts, r=r_rad)
    covered = np.array([len(idx) > 0 for idx in within], dtype=bool)
    zip_universe[f"covered_{ctype}_50mi"] = covered


In [37]:

# ==============================
# Income quartiles (population-weighted, national)
# ==============================
def weighted_quantile(values, weights, quantiles):
    v = np.asarray(values); w = np.asarray(weights)
    idx = np.argsort(v); v=v[idx]; w=w[idx]
    cw = np.cumsum(w) - 0.5*w
    cw = cw / np.sum(w)
    return np.interp(quantiles, cw, v)

mask_inc = zip_universe["mhi"].notna() & zip_universe["population"].notna()
if mask_inc.any():
    q1, q2, q3 = weighted_quantile(zip_universe.loc[mask_inc,"mhi"], zip_universe.loc[mask_inc,"population"], [0.25,0.50,0.75])
else:
    q1=q2=q3=np.nan

zip_universe["inc_quartile"] = pd.cut(zip_universe["mhi"],
                                     bins=[-np.inf, q1, q2, q3, np.inf],
                                     labels=["Q1 (Lowest)", "Q2", "Q3", "Q4 (Highest)"])
zip_universe["inc_quartile"].value_counts(dropna=False)


inc_quartile
Q1 (Lowest)     12888
Q2               8345
Q3               6785
Q4 (Highest)     4903
NaN               223
Name: count, dtype: int64

In [38]:

# ==============================
# Coverage shares — overall and by strata
# ==============================
def pop_share(df: pd.DataFrame, covered_col: str, weight_col: str) -> float:

    if covered_col not in df or weight_col not in df: 
        return float("nan")
    num = (df[covered_col] * df[weight_col]).sum(skipna=True)
    den = df[weight_col].sum(skipna=True)
    return float(num/den) if den > 0 else float("nan")

def covered_population_millions(df: pd.DataFrame, covered_col: str) -> float:
    num = (df[covered_col] * df["population"]).sum(skipna=True)
    return float(num / 1_000_000.0)

def race_eth_share(df: pd.DataFrame, covered_col: str, num_col: str, den_col: str) -> float:

    num = (df[covered_col] * df[num_col]).sum(skipna=True)
    den = df[den_col].sum(skipna=True)
    return float(num/den) if den > 0 else float("nan")


In [41]:
# ==============================
# Build Page-2 style table (robust + fixed subgroup denominators)
# ==============================
import numpy as np
import pandas as pd

# ---- Choose which coverage radius column to use (must match your coverage cell) ----
COVERAGE_SUFFIX = "50mi"   # e.g., "30mi", "60mi", "120mi"

# --- Safety net: ensure every covered_* column exists, default False ---
for key in CANCER_LABELS.keys():
    col = f"covered_{key}_{COVERAGE_SUFFIX}" if key != "any_gi" else f"covered_any_gi_{COVERAGE_SUFFIX}"
    if col not in zip_universe.columns:
        zip_universe[col] = False

# --- Helpers (defensive) ---
def pop_share(df: pd.DataFrame, covered_col: str, weight_col: str = "population") -> float:
    if covered_col not in df.columns or weight_col not in df.columns or len(df) == 0:
        return float("nan")
    w = pd.to_numeric(df[weight_col], errors="coerce").fillna(0.0)
    c = df[covered_col].astype(bool)
    den = w.sum()
    if den <= 0:
        return float("nan")
    num = (c * w).sum()
    return float(num / den)

def covered_population_millions(df: pd.DataFrame, covered_col: str) -> float:
    if covered_col not in df.columns or "population" not in df.columns or len(df) == 0:
        return float("nan")
    w = pd.to_numeric(df["population"], errors="coerce").fillna(0.0)
    c = df[covered_col].astype(bool)
    num = (c * w).sum()
    return float(num / 1_000_000.0)

def subpop_covered_share(df: pd.DataFrame, covered_col: str, subpop_col: str) -> float:
    """
    Correct subgroup metric:
    % of the subgroup that is covered = sum(covered * subgroup_pop) / sum(subgroup_pop)
    """
    if covered_col not in df.columns or subpop_col not in df.columns or len(df) == 0:
        return float("nan")
    w = pd.to_numeric(df[subpop_col], errors="coerce").fillna(0.0)
    den = w.sum()
    if den <= 0:
        return float("nan")
    c = df[covered_col].astype(bool)
    num = (w[c]).sum()
    return float(num / den)

def _counts_for(key: str) -> tuple[int, int]:
    if key == "any_gi":
        n_trials = int(trials_filtered["nct_id"].nunique()) if "nct_id" in trials_filtered.columns else 0
        n_sites  = int(trial_sites["zip"].nunique())         if "zip"    in trial_sites.columns    else 0
        return n_trials, n_sites
    if "cancer_type" in trials_filtered.columns:
        tf = trials_filtered.loc[trials_filtered["cancer_type"] == key]
        n_trials = int(tf["nct_id"].nunique()) if "nct_id" in tf.columns else 0
    else:
        n_trials = 0
    if "cancer_type" in trial_sites.columns:
        ts = trial_sites.loc[trial_sites["cancer_type"] == key]
        n_sites = int(ts["zip"].nunique()) if "zip" in ts.columns else 0
    else:
        n_sites = 0
    return n_trials, n_sites

def _covered_col_for(key: str) -> str:
    return f"covered_any_gi_{COVERAGE_SUFFIX}" if key == "any_gi" else f"covered_{key}_{COVERAGE_SUFFIX}"

# --- Build rows ---
rows = []
for key, label in CANCER_LABELS.items():
    covered_col = _covered_col_for(key)
    n_trials, n_sites = _counts_for(key)

    # Overall population share & covered population
    pop_share_all = pop_share(zip_universe, covered_col, "population")
    covered_pop_m = covered_population_millions(zip_universe, covered_col)

    # Urban/Rural shares
    if "rural_urban" in zip_universe.columns:
        pop_share_urban = pop_share(zip_universe[zip_universe["rural_urban"] == "Urban"], covered_col, "population")
        pop_share_rural = pop_share(zip_universe[zip_universe["rural_urban"] == "Rural"],  covered_col, "population")
    else:
        pop_share_urban = float("nan")
        pop_share_rural = float("nan")

    # Income quartiles
    if "inc_quartile" in zip_universe.columns:
        q1_df = zip_universe[zip_universe["inc_quartile"] == "Q1 (Lowest)"]
        q4_df = zip_universe[zip_universe["inc_quartile"] == "Q4 (Highest)"]
        pop_share_q1 = pop_share(q1_df, covered_col, "population")
        pop_share_q4 = pop_share(q4_df, covered_col, "population")
    else:
        pop_share_q1 = float("nan")
        pop_share_q4 = float("nan")

    # Race/ethnicity shares (correct denominators)
    hisp_share  = subpop_covered_share(zip_universe, covered_col, "pop_hispanic")
    black_share = subpop_covered_share(zip_universe, covered_col, "pop_black")

    rows.append({
        "Cancer Type": label,
        "No. Trials (NCT IDs)": n_trials if n_trials > 0 else np.nan,
        "No. Sites (ZIPs)": n_sites if n_sites > 0 else np.nan,
        "% Population Covered": pop_share_all * 100.0 if not np.isnan(pop_share_all) else np.nan,
        "Covered Population (M)": covered_pop_m if not np.isnan(covered_pop_m) else np.nan,
        "% Patients Covered (SEER Incidence)": np.nan,  # placeholder (No SEER)
        "% Urban Pop Covered": pop_share_urban * 100.0 if not np.isnan(pop_share_urban) else np.nan,
        "% Rural Pop Covered": pop_share_rural * 100.0 if not np.isnan(pop_share_rural) else np.nan,
        "% Lowest Income Quartile Covered": pop_share_q1 * 100.0 if not np.isnan(pop_share_q1) else np.nan,
        "% Highest Income Quartile Covered": pop_share_q4 * 100.0 if not np.isnan(pop_share_q4) else np.nan,
        "% Hispanic Pop Covered": hisp_share * 100.0 if not np.isnan(hisp_share) else np.nan,
        "% Black Pop Covered": black_share * 100.0 if not np.isnan(black_share) else np.nan,
    })

table = pd.DataFrame(rows)

# Nicely formatted version
fmt = table.copy()
for col in [
    "% Population Covered",
    "% Patients Covered (SEER Incidence)",
    "% Urban Pop Covered",
    "% Rural Pop Covered",
    "% Lowest Income Quartile Covered",
    "% Highest Income Quartile Covered",
    "% Hispanic Pop Covered",
    "% Black Pop Covered",
]:
    if col in fmt.columns:
        fmt[col] = fmt[col].map(lambda x: f"{x:.1f}%" if pd.notna(x) else "NA")

if "Covered Population (M)" in fmt.columns:
    fmt["Covered Population (M)"] = fmt["Covered Population (M)"].map(lambda x: f"{x:.2f}" if pd.notna(x) else "NA")

fmt


,Cancer Type,No. Trials (NCT IDs),No. Sites (ZIPs),% Population Covered,Covered Population (M),% Patients Covered (SEER Incidence),% Urban Pop Covered,% Rural Pop Covered,% Lowest Income Quartile Covered,% Highest Income Quartile Covered,% Hispanic Pop Covered,% Black Pop Covered
0,Any GI,969.0,1933.0,95.9%,320.98,NA,98.2%,84.2%,91.7%,99.6%,95.9%,96.8%
1,Colorectal,430.0,1518.0,94.5%,316.34,NA,97.5%,79.7%,89.2%,99.4%,95.3%,95.5%
2,Pancreatic,338.0,1240.0,92.7%,310.19,NA,95.9%,76.3%,84.9%,98.9%,91.9%,93.1%
3,Gastric,220.0,1337.0,93.4%,312.58,NA,96.6%,77.3%,86.6%,99.0%,94.3%,92.9%
4,Esophageal,139.0,1031.0,89.3%,298.99,NA,93.2%,70.0%,78.6%,98.2%,88.8%,89.3%
5,Liver (HCC),157.0,390.0,79.0%,264.58,NA,86.1%,44.1%,67.0%,94.8%,83.6%,84.9%
6,Cholangiocarcinoma,66.0,396.0,75.4%,252.45,NA,82.2%,41.6%,60.5%,92.1%,76.4%,79.9%
7,Other GI,NaN,NaN,0.0%,0.00,NA,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%


In [ ]:

# ==============================
# Save outputs
# ==============================
raw_csv  = os.path.join(OUT_DIR, "page2_table_complete_raw.csv")
fmt_csv  = os.path.join(OUT_DIR, "page2_table_complete_formatted.csv")
xlsx_out = os.path.join(OUT_DIR, "page2_table_complete.xlsx")

table.to_csv(raw_csv, index=False)
fmt.to_csv(fmt_csv, index=False)

with pd.ExcelWriter(xlsx_out, engine="xlsxwriter") as w:
    fmt.to_excel(w, sheet_name="Page2_Table", index=False)

print("Saved:")
print(" -", raw_csv)
print(" -", fmt_csv)
print(" -", xlsx_out)



### (Optional) Add SEER incidence to compute **% Patients Covered (SEER Incidence)**

If you later have SEER incidence counts by cancer type:

1. Load a CSV with columns: `Cancer Type` (matching labels above) and `Incidence` (national counts).
2. For each row, compute patients covered as: **(Incidence living in ZIPs ≤50mi of ≥1 site) / (Total Incidence)**.
3. You can approximate by applying the **% Population Covered** to the incidence (if you assume incidence distributed proportionally to population). A more precise estimate uses SEER ZIP- or county-level incidence mapped to ZIPs.

> For now, the column is left as `NA`.


In [ ]:

# ==============================
# Quick sanity checks (prints)
# ==============================
print("Any GI — % Pop Covered (50mi):",
      (table.loc[table["Cancer Type"]=="Any GI", "% Population Covered"].values[0]
       if (table["Cancer Type"]=="Any GI").any() else "NA"),
     )
print("Any GI — No. Trials (NCT):",
      (table.loc[table["Cancer Type"]=="Any GI", "No. Trials (NCT IDs)"].values[0]
       if (table["Cancer Type"]=="Any GI").any() else "NA"),
     )
print("Any GI — No. Sites (ZIPs):",
      (table.loc[table["Cancer Type"]=="Any GI", "No. Sites (ZIPs)"].values[0]
       if (table["Cancer Type"]=="Any GI").any() else "NA"),
     )
